# Tech Challenge Fase 3 — Análise Exploratória de Dados

**Objetivo:** Entender a base de dados da camada Gold (Fase 2) para apoiar as decisões de modelagem.

**Fonte:** S3 `tech-challenge-alfabetizacao-01/gold/` — 6 tabelas Parquet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 1. Carregamento dos Dados

In [ ]:
indicador    = pd.read_parquet('../data/gold/indicador_municipio/indicador_municipio.parquet')
proficiencia = pd.read_parquet('../data/gold/proficiencia_municipio/proficiencia_municipio.parquet')
evolucao_uf  = pd.read_parquet('../data/gold/evolucao_temporal_uf/evolucao_temporal_uf.parquet')
ranking_uf   = pd.read_parquet('../data/gold/ranking_uf/ranking_uf.parquet')
risco        = pd.read_parquet('../data/gold/municipios_risco/municipios_risco.parquet')
comparativo  = pd.read_parquet('../data/gold/comparativo_nacional/comparativo_nacional.parquet')

print('indicador_municipio:  ', indicador.shape)
print('proficiencia_municipio:', proficiencia.shape)
print('evolucao_temporal_uf:  ', evolucao_uf.shape)
print('ranking_uf:            ', ranking_uf.shape)
print('municipios_risco:      ', risco.shape)
print('comparativo_nacional:  ', comparativo.shape)

## 2. Tabela Principal — `indicador_municipio`

Esta é a tabela com maior granularidade (município × ano × série × rede) e será a base do modelo.

In [ ]:
indicador.head()

In [ ]:
indicador.dtypes

In [ ]:
indicador.describe()

## 3. Análise de Valores Nulos

In [ ]:
nulls = indicador.isnull().sum()
nulls[nulls > 0]

In [ ]:
# Os nulls nas proporcoes de nivel estao 100% concentrados no ano 2023
# A fonte (SAEB/ANA) nao disponibilizou esse detalhamento para 2023
null_mask = indicador['proporcao_aluno_nivel_0'].isnull()
print('Ano dos registros com null em proporcao_nivel:')
print(indicador[null_mask]['ano'].value_counts())
print()
print('Ano dos registros sem null:')
print(indicador[~null_mask]['ano'].value_counts())

**Conclusão:** Os nulls não são erro de dados — é limitação da fonte. As colunas `proporcao_aluno_nivel_*` só existem para 2024.

**Decisão de modelagem (a definir):**
- Opção A: usar apenas dados de 2024 (12.448 registros, sem nulls)
- Opção B: usar todos os anos, excluindo as 4 colunas de proporção
- Opção C: imputar as proporções de 2023 com mediana por UF

## 4. Variável Target — `categoria_risco`

O challenge pede classificação binária (alfabetizado / não alfabetizado).
`categoria_risco` tem 4 categorias — precisamos definir o mapeamento.

In [ ]:
print(indicador['categoria_risco'].value_counts())
print()
print(indicador.groupby('categoria_risco')['taxa_alfabetizacao'].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuicao das categorias
indicador['categoria_risco'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribuição de categoria_risco')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Taxa de alfabetizacao por categoria
indicador.boxplot(column='taxa_alfabetizacao', by='categoria_risco', ax=axes[1])
axes[1].set_title('Taxa de alfabetização por categoria')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Proposta de target binario:
# meta_atingida = 1 (alfabetizado)
# critico + alto + moderado = 0 (nao alfabetizado / em risco)
indicador['target'] = (indicador['categoria_risco'] == 'meta_atingida').astype(int)
print('Distribuicao do target binario:')
print(indicador['target'].value_counts())
print()
print(f'Balanceamento: {indicador["target"].mean():.1%} positivos')

## 5. Distribuições das Features Numéricas

In [ ]:
num_cols = ['taxa_alfabetizacao', 'media_portugues', 'meta_mun_2030',
            'meta_uf_2030', 'gap_meta_municipio_2030', 'gap_meta_uf_2030']

indicador[num_cols].hist(bins=40, figsize=(14, 8))
plt.suptitle('Distribuição das Features Numéricas')
plt.tight_layout()
plt.show()

## 6. Correlações

In [ ]:
corr = indicador[num_cols + ['target']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

## 7. Análise por UF

In [ ]:
media_uf = indicador.groupby('sigla_uf')['taxa_alfabetizacao'].mean().sort_values()

plt.figure(figsize=(14, 6))
media_uf.plot(kind='bar', color='steelblue')
plt.title('Taxa média de alfabetização por UF')
plt.ylabel('Taxa média')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Hipóteses Analíticas

*(preencher após análise visual)*

1. **H1:** Municípios com maior `media_portugues` têm maior probabilidade de atingir a meta
2. **H2:** O `gap_meta_uf_2030` é um preditor forte do risco educacional
3. **H3:** Existem padrões regionais claros — regiões Norte e Nordeste concentram maior risco
4. **H4:** A rede de ensino (pública/privada) influencia significativamente a taxa de alfabetização

*(validar as hipóteses nas células acima e documentar aqui as conclusões)*

## 9. Decisões para a Modelagem

*(preencher após concluir EDA)*

- **Target:** `categoria_risco` → binário (`meta_atingida` = 1, demais = 0)
- **Tratamento dos nulls:** TBD (ver Seção 3)
- **Features a incluir:** TBD
- **Features a excluir:** TBD (ex: leakage potencial em `gap_meta_*`)
- **Desbalanceamento:** TBD — verificar se necessita SMOTE ou class_weight